## imports

### reload

In [1]:
%load_ext autoreload
%autoreload 2

### imports dependencies

In [2]:
import json
import sys
import chromadb
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

# ingestion
from app.ingestion.chunker import chunk_docs
from app.ingestion.store import store_documents
from app.schemas import Document
from app.rag.embeddings import MyEmbeddingFunctionSentenceTransformer

# rag 
from app.rag.pipeline import rag_pipeline

d:\Programs\anaconda3\envs\rag_py3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open("../dataset/machine_learning_knowledge.json") as f:
    docs = json.load(f)

    print(docs[0])

{'id': 'doc1', 'text': 'Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition. The performance of a machine learning model typically improves as it is exposed to more high-quality data.'}


In [4]:
documents = [Document(id=doc["id"], content=doc["text"]) for doc in docs]

### setup storage

In [5]:
    client = chromadb.PersistentClient(path="../chroma_db")
    embedding_function = MyEmbeddingFunctionSentenceTransformer()
    collection = client.get_or_create_collection(
        name="rag_collection",
        embedding_function=embedding_function
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1189.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## ingest

In [6]:
docs = chunk_docs(documents)
store_documents(collection, docs)

## rag pipeline

In [7]:
question = "What is machine learning and what is its main goal?"

In [9]:
answer, doc_result = rag_pipeline(question, collection)
print(answer)
print()
for doc in doc_result:
    print(doc.id)
    print(doc.content)
    print(doc.distance)
    print("=========================")

Machine Learning is a subset of Artificial Intelligence (AI) that involves training algorithms to make predictions, decisions or take actions based on data. This allows machines to improve their performance over time without being explicitly programmed.

The main goal of Machine Learning is to enable computers to learn from data and improve their performance in specific tasks through experience and iteration.

In simpler terms, machine learning enables computers to:

1. Learn patterns and relationships within data
2. Make predictions or decisions based on that data
3. Improve their performance over time as they receive more data

The ultimate goal of Machine Learning is often to create autonomous systems that can improve their own performance, making them capable of solving complex problems and making decisions without human intervention.

However, the field of machine learning has some limitations, such as:

* It requires large amounts of data
* It's not yet fully understood how human